# fhir4ds.viewdef

SQL-on-FHIR v2 ViewDefinition generator. This notebook demonstrates how to define portable FHIR views and execute them in DuckDB.

In [ ]:
# Install the unified package
%pip install fhir4ds-v2

In [ ]:
import json
import fhir4ds
from fhir4ds import generate_view_sql

connection = fhir4ds.create_connection()
connection.execute("CREATE TABLE patients (resource JSON)")

patient = {
    "resourceType": "Patient",
    "id": "p1",
    "name": [
        {"given": ["John"], "family": "Doe"},
        {"given": ["Jane"], "family": "Smith"},
    ],
}
connection.execute("INSERT INTO patients VALUES (?)", [json.dumps(patient)])

### Collection Flattening with `forEach` 
ViewDefinitions can flatten arrays into rows using `forEach`. In DuckDB, this translates to efficient `CROSS JOIN LATERAL` / `UNNEST` operations.

In [ ]:
view_definition = {
    "resource": "Patient",
    "select": [
        {"column": [{"path": "id", "name": "patient_id"}]},
        {
            "forEach": "name",
            "column": [
                {"path": "family", "name": "family_name"},
                {"path": "given.first()", "name": "given_name"}
            ]
        }
    ],
}

sql = generate_view_sql(json.dumps(view_definition))
connection.execute(sql).df()